# # PRE_08 — Horas y millas por conductor con SQLite
#
# **Qué se hace:** se cargan `drivers.csv` y `timesheet.csv` en una base SQLite y la
# agregación por conductor se resuelve con SQL. pandas solo mueve los datos y dibuja.
#
# **Entregables** (`submission/`):
# - `summary.csv`: una fila por conductor (semanas, horas y millas totales y promedio, millas por hora).
# - `top10_drivers.png`: barras horizontales con los 10 conductores de más millas.
#
# **Decisiones y por qué:**
# - `LEFT JOIN` desde `drivers`: un conductor sin registros semanales no debe desaparecer del reporte.
# - `ssn` y `location` quedan fuera: son identificadores directos y el reporte es agregado.
# - `NULLIF` en el divisor: evita la división entre cero si un conductor tiene 0 horas.

In [1]:
import sqlite3
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # sin ventana: el gráfico se guarda a archivo
import matplotlib.pyplot as plt
import pandas as pd

# Funciona tanto desde la carpeta de la actividad como desde notebooks/ (Jupyter)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
SUBMISSION = ROOT / "submission"
SUBMISSION.mkdir(exist_ok=True)

drivers = pd.read_csv(ROOT / "data" / "drivers.csv")
timesheet = pd.read_csv(ROOT / "data" / "timesheet.csv")
print(drivers.shape, timesheet.shape)
print("Nulos drivers:", int(drivers.isna().sum().sum()), "| nulos timesheet:", int(timesheet.isna().sum().sum()))
print("Registros duplicados (driverId, week):", int(timesheet.duplicated(["driverId", "week"]).sum()))
print("Conductores sin registros:", sorted(set(drivers.driverId) - set(timesheet.driverId)))

(34, 6) (1768, 4)
Nulos drivers: 0 | nulos timesheet: 0
Registros duplicados (driverId, week): 0
Conductores sin registros: []


# ## Base SQLite y consulta
# La base se crea en memoria: no hace falta persistirla, se reconstruye en cada ejecución.

In [2]:
SUMMARY_QUERY = """
    SELECT
        d.driverId                                   AS driverId,
        d.name                                       AS name,
        d.certified                                  AS certified,
        d."wage-plan"                                AS "wage-plan",
        COUNT(t.week)                                AS weeks,
        COALESCE(SUM(t."hours-logged"), 0)           AS total_hours,
        COALESCE(SUM(t."miles-logged"), 0)           AS total_miles,
        ROUND(AVG(t."hours-logged"), 1)              AS mean_hours,
        ROUND(AVG(t."miles-logged"), 1)              AS mean_miles,
        ROUND(
            COALESCE(SUM(t."miles-logged"), 0) * 1.0
            / NULLIF(SUM(t."hours-logged"), 0), 2
        )                                            AS miles_per_hour
    FROM drivers AS d
    LEFT JOIN timesheet AS t
        ON d.driverId = t.driverId
    GROUP BY d.driverId, d.name, d.certified, d."wage-plan"
    ORDER BY total_miles DESC
"""

connection = sqlite3.connect(":memory:")
drivers.to_sql("drivers", connection, index=False)
timesheet.to_sql("timesheet", connection, index=False)
summary = pd.read_sql_query(SUMMARY_QUERY, connection)
connection.close()
summary.head(10)

,driverId,name,certified,wage-plan,weeks,total_hours,total_miles,mean_hours,mean_miles,miles_per_hour
0,11,Jamie Engesser,N,miles,52,3642,179300,70.0,3448.1,49.23
1,10,George Vetticaden,N,miles,52,3232,147150,62.2,2829.8,45.53
2,33,Sridhara Sabbella,Y,hours,52,2759,139285,53.1,2678.6,50.48
3,25,Jean-Philippe Playe,Y,hours,52,2723,139180,52.4,2676.5,51.11
4,39,David Kaiser,Y,hours,52,2745,138788,52.8,2669.0,50.56
5,15,Rohit Bakshi,Y,hours,52,2734,138750,52.6,2668.3,50.75
6,35,Emil Siemes,Y,hours,52,2728,138727,52.5,2667.8,50.85
7,21,Jeff Markham,Y,hours,52,2751,138719,52.9,2667.7,50.42
8,41,Greg Phillips,Y,hours,52,2723,138407,52.4,2661.7,50.83
9,29,Teddy Choi,Y,hours,52,2760,138255,53.1,2658.8,50.09


# ## Validación
# Los totales del resumen deben cuadrar con los del archivo original y no debe haber datos personales.

In [3]:
assert len(summary) == len(drivers) == summary.driverId.nunique()
assert summary.total_hours.sum() == timesheet["hours-logged"].sum()
assert summary.total_miles.sum() == timesheet["miles-logged"].sum()
assert summary.weeks.sum() == len(timesheet)
assert not {"ssn", "location"} & set(summary.columns)

summary.to_csv(SUBMISSION / "summary.csv", index=False)
print(f"{len(summary)} conductores | {summary.total_hours.sum():,} horas | {summary.total_miles.sum():,} millas")

34 conductores | 94,201 horas | 4,722,737 millas


# ## Top 10 por millas

In [4]:
top = summary.nlargest(10, "total_miles").iloc[::-1]  # invertido: el mayor queda arriba

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(top["name"], top["total_miles"], color="#4C78A8")
ax.set_xlabel("Millas recorridas en el año")
ax.set_title("Top 10 conductores por millas recorridas")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.set_xlim(0, top["total_miles"].max() * 1.12)
for bar, value in zip(bars, top["total_miles"]):  # valor sobre la barra
    ax.text(value * 1.01, bar.get_y() + bar.get_height() / 2, f"{value:,}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(SUBMISSION / "top10_drivers.png", dpi=150)
plt.show()

/var/folders/gj/yjfyzlqd4fxcmlnj64g5s9bw0000gn/T/ipykernel_5372/4269833434.py:14: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


# ## Hallazgos

In [5]:
by_plan = summary.groupby("wage-plan").agg(
    conductores=("driverId", "count"),
    millas_promedio=("total_miles", "mean"),
    horas_promedio=("total_hours", "mean"),
    millas_por_hora=("miles_per_hour", "mean"),
).round(2)
print(by_plan)
lider = summary.iloc[0]
print(f"\nLíder: {lider['name']} con {lider['total_miles']:,} millas "
      f"({lider['total_miles'] / summary.total_miles.median():.2f}x la mediana)")

           conductores  millas_promedio  horas_promedio  millas_por_hora
wage-plan                                                               
hours               32        137383.97         2728.97            50.35
miles                2        163225.00         3437.00            47.38

Líder: Jamie Engesser con 179,300 millas (1.30x la mediana)


# - 34 conductores y 1.767 registros semanales; el top 10 reúne ~30 % de las millas totales.
# - El líder (Jamie Engesser, 179.300 millas) está ~1,3x sobre la mediana (137.540).
# - Los 2 conductores del plan `miles` recorren más (~163 mil millas) y manejan más horas (~3.437),
#   pero su rendimiento es **menor** (47,4 mi/h vs 50,3 del plan `hours`): más horas, no más rápido.
# - El `LEFT JOIN` + `COALESCE` deja el reporte robusto ante conductores sin registros.